In [ ]:
# Auto-detect runtime, set env, and launch DETReg epithelial matrix from Jupyter.

import os, sys, subprocess, shlex, pathlib, glob, os.path as op

SCRIPT = "TrainDETReg_SOLO_SingleClass_STATIC_Ucloud.py"


def _detect_runtime_profile() -> str:
    forced = os.environ.get("DETREG_RUNTIME_PROFILE", "auto").strip().lower()
    if forced in {"ucloud", "local"}:
        return forced
    if forced not in {"", "auto"}:
        raise ValueError("DETREG_RUNTIME_PROFILE must be one of: auto, ucloud, local")
    if os.name != "nt" and pathlib.Path("/work").exists():
        has_member_files = any(pathlib.Path("/work").glob("Member Files:*"))
        has_hash_user = any(pathlib.Path("/work").glob("*#*"))
        if has_member_files or has_hash_user:
            return "ucloud"
    return "local"


def _detect_ucloud_user_base() -> str:
    work = pathlib.Path("/work")
    if not work.exists():
        return ""
    member = sorted(work.glob("Member Files:*"))
    if member:
        return member[0].name
    hashed = sorted([p for p in work.glob("*#*") if p.is_dir()])
    return hashed[0].name if hashed else ""


def _latest_dataset_dir(root: pathlib.Path, token: str) -> str:
    cands = sorted([p for p in root.glob(f"*{token}*") if p.is_dir()])
    return str(cands[-1]) if cands else ""


RUNTIME_PROFILE = _detect_runtime_profile()
os.environ["DETREG_RUNTIME_PROFILE"] = RUNTIME_PROFILE

if RUNTIME_PROFILE == "ucloud":
    PROJECT_DIR = pathlib.Path(os.environ.get("PROJECT_DIR", "/work/projects/myproj/SOLO_Supervised_RFDETR")).resolve()
else:
    PROJECT_DIR = pathlib.Path(os.environ.get("PROJECT_DIR", str(pathlib.Path.cwd() / "SOLO_Supervised_RFDETR"))).resolve()

REPO_ROOT = PROJECT_DIR.parent
print("DETREG_RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("PROJECT_DIR =", PROJECT_DIR)

if RUNTIME_PROFILE == "ucloud":
    UCLOUD_USER_BASE = _detect_ucloud_user_base()
    DATASET_ROOT_DEFAULT = pathlib.Path(os.environ.get("STAT_DATASETS_ROOT", str(PROJECT_DIR / "Stat_Dataset")))
    if UCLOUD_USER_BASE:
        IMAGES_FALLBACK_DEFAULT = pathlib.Path(f"/work/{UCLOUD_USER_BASE}/CellScanData/Zoom10x - Quality Assessment_Cleaned")
        OUTPUT_ROOT_DEFAULT = pathlib.Path(f"/work/{UCLOUD_USER_BASE}/RFDETR_SOLO_OUTPUT/DETREG_EPI")
        PRETRAIN_CKPT_DEFAULT = pathlib.Path(f"/work/{UCLOUD_USER_BASE}/DETReg_Checkpoints/checkpoint_coco.pth")
    else:
        IMAGES_FALLBACK_DEFAULT = REPO_ROOT / "CellScanData" / "Zoom10x - Quality Assessment_Cleaned"
        OUTPUT_ROOT_DEFAULT = PROJECT_DIR / "RFDETR_SOLO_OUTPUT" / "DETREG_EPI"
        PRETRAIN_CKPT_DEFAULT = REPO_ROOT / "DETReg_Checkpoints" / "checkpoint_coco.pth"
    DETREG_REPO_DEFAULT = pathlib.Path(os.environ.get("DETREG_REPO_DIR", "/work/projects/myproj/DETReg"))
else:
    DATASET_ROOT_DEFAULT = pathlib.Path(os.environ.get("STAT_DATASETS_ROOT", str(PROJECT_DIR / "Stat_Dataset")))
    IMAGES_FALLBACK_DEFAULT = pathlib.Path(r"D:\PHD\PhdData\CellScanData\Zoom10x - Quality Assessment_Cleaned")
    OUTPUT_ROOT_DEFAULT = pathlib.Path(os.environ.get("OUTPUT_ROOT", str(PROJECT_DIR / "RFDETR_SOLO_OUTPUT" / "DETREG_EPI_local")))
    PRETRAIN_CKPT_DEFAULT = pathlib.Path(os.environ.get("DETREG_PRETRAIN_CKPT", str(REPO_ROOT / "DETReg_Checkpoints" / "checkpoint_coco.pth")))
    DETREG_REPO_DEFAULT = pathlib.Path(os.environ.get("DETREG_REPO_DIR", str(REPO_ROOT / "_external_DETReg_ref")))

DATASET_EPI_DEFAULT = os.environ.get("DATASET_EPI", "").strip() or _latest_dataset_dir(DATASET_ROOT_DEFAULT, "SquamousEpithelialCell_OVR")

# Requested run matrix: epithelial only, 50%/100%, 3 seeds.
os.environ["DATASET_EPI"] = DATASET_EPI_DEFAULT
os.environ["STAT_DATASETS_ROOT"] = str(DATASET_ROOT_DEFAULT)
os.environ["IMAGES_FALLBACK_ROOT"] = str(pathlib.Path(os.environ.get("IMAGES_FALLBACK_ROOT", str(IMAGES_FALLBACK_DEFAULT))))
os.environ["OUTPUT_ROOT"] = str(pathlib.Path(os.environ.get("OUTPUT_ROOT", str(OUTPUT_ROOT_DEFAULT))))
os.environ["DETREG_REPO_DIR"] = str(pathlib.Path(os.environ.get("DETREG_REPO_DIR", str(DETREG_REPO_DEFAULT))))
os.environ["DETREG_PRETRAIN_CKPT"] = str(pathlib.Path(os.environ.get("DETREG_PRETRAIN_CKPT", str(PRETRAIN_CKPT_DEFAULT))))
os.environ.setdefault("DETREG_PRETRAIN_URL", "https://github.com/amirbar/DETReg/releases/download/1.0.0/checkpoint_coco.pth")
os.environ.setdefault("DETREG_AUTO_DOWNLOAD_PRETRAIN", "1")

os.environ["DETREG_TRAIN_FRACTIONS"] = "0.5,1.0"
os.environ["DETREG_SEEDS"] = "42,43,44"
os.environ["DETREG_FRACTION_SEED"] = "42"
os.environ.setdefault("DETREG_TARGET_NAME", "Squamous Epithelial Cell")
os.environ.setdefault("DETREG_MODEL", "deformable_detr")
os.environ.setdefault("DETREG_EPOCHS", "50")
os.environ.setdefault("DETREG_BATCH_SIZE", "4")
os.environ.setdefault("DETREG_LR_DROP", "40")
os.environ.setdefault("DETREG_NUM_WORKERS", "8")
os.environ.setdefault("DETREG_DATA_LINK_MODE", "symlink")
os.environ.setdefault("DETREG_EVAL_EVERY", "1")

print("DATASET_EPI =", os.environ["DATASET_EPI"])
print("DETREG_REPO_DIR =", os.environ["DETREG_REPO_DIR"])
print("DETREG_PRETRAIN_CKPT =", os.environ["DETREG_PRETRAIN_CKPT"])
print("DETREG_PRETRAIN_URL =", os.environ.get("DETREG_PRETRAIN_URL"))
print("DETREG_AUTO_DOWNLOAD_PRETRAIN =", os.environ.get("DETREG_AUTO_DOWNLOAD_PRETRAIN"))
print("DETREG_TRAIN_FRACTIONS =", os.environ["DETREG_TRAIN_FRACTIONS"])
print("DETREG_SEEDS =", os.environ["DETREG_SEEDS"])
print("DETREG_FRACTION_SEED =", os.environ["DETREG_FRACTION_SEED"])
print("IMAGES_FALLBACK_ROOT =", os.environ["IMAGES_FALLBACK_ROOT"])
print("OUTPUT_ROOT =", os.environ["OUTPUT_ROOT"])

if not pathlib.Path(os.environ["DATASET_EPI"]).exists():
    raise FileNotFoundError(f"DATASET_EPI does not exist: {os.environ['DATASET_EPI']}")
if not pathlib.Path(os.environ["IMAGES_FALLBACK_ROOT"]).exists():
    raise FileNotFoundError(f"IMAGES_FALLBACK_ROOT does not exist: {os.environ['IMAGES_FALLBACK_ROOT']}")
if not pathlib.Path(os.environ["DETREG_REPO_DIR"]).exists():
    raise FileNotFoundError(
        f"DETREG_REPO_DIR does not exist: {os.environ['DETREG_REPO_DIR']}\n"
        "Clone DETReg first, e.g.: git clone https://github.com/amirbar/DETReg /work/projects/myproj/DETReg"
    )

script_path = PROJECT_DIR / SCRIPT
if not script_path.exists():
    raise FileNotFoundError(f"Script not found: {script_path}")

cmd = [sys.executable, "-u", str(script_path)]
print("\n[LAUNCH]")
print(" cwd:", PROJECT_DIR)
print(" cmd:", " ".join(shlex.quote(x) for x in cmd))

proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for _line in proc.stdout:
    print(_line, end="")
ret = proc.wait()
if ret != 0:
    raise subprocess.CalledProcessError(ret, cmd)


